# Expanded Hallucination Evaluation — GPU stages (Colab)

Runs the SLOW detectors of the expanded evaluation (4,000-row suite:
2,600 standard + 1,400 hard) on a GPU: **naive NLI, HHEM, LettuceDetect,
SummaC — all FULL, no sampling**. Our checker also re-runs here (fast on
GPU) as an independent cross-check of the local numbers.

**Runtime → Change runtime type → T4 GPU** before starting.

**Before running, build the bundle locally:**
```
venv\\Scripts\\python.exe m3_implementation\\test_result\\hallucination_result\\expanded_eval\\make_expanded_colab_bundle.py
```
Upload `expanded_colab_bundle.zip` when cell 3 asks.

At the end, cell 8 downloads `colab_results.zip` — hand that back for
integration. Expected total runtime on T4: **~1–1.5 h** (vs ~10 h on CPU).

In [ ]:
# 1 — Dependencies (summac needs --no-deps: its pins are stale)
!pip -q install sentence-transformers httpx python-dotenv lettucedetect nltk sentencepiece
!pip -q install --no-deps summac
import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
print('deps OK')

In [ ]:
# 2 — CRITICAL: match the local configuration exactly
import os
os.environ['NLI_CONTRADICTION_THRESHOLD'] = '0.70'   # local .env value
os.environ['NLI_MODEL_NAME'] = 'cross-encoder/nli-deberta-v3-base'
print('config set: threshold 0.70')

In [ ]:
# 3 — Upload and extract expanded_colab_bundle.zip
import zipfile
from google.colab import files
up = files.upload()
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall('.')
EE = 'm3_implementation/test_result/hallucination_result/expanded_eval'
assert os.path.exists(f'{EE}/labeled_test_set_expanded.jsonl')
assert os.path.exists(f'{EE}/hard_set/labeled_hard_set.jsonl')
print('bundle OK')

In [ ]:
# 4 — Ours + naive NLI, FULL, standard set (2600)
!python m3_implementation/test_result/hallucination_result/run_detector_eval.py \
  --test-set m3_implementation/test_result/hallucination_result/expanded_eval/labeled_test_set_expanded.jsonl \
  --skip-llm \
  --out m3_implementation/test_result/hallucination_result/expanded_eval/colab_results_standard_ours_naive.json

In [ ]:
# 5 — Ours + naive NLI, FULL, hard set (1400)
!python m3_implementation/test_result/hallucination_result/run_detector_eval.py \
  --test-set m3_implementation/test_result/hallucination_result/expanded_eval/hard_set/labeled_hard_set.jsonl \
  --skip-llm \
  --out m3_implementation/test_result/hallucination_result/expanded_eval/colab_results_hard_ours_naive.json

In [ ]:
# 6 — HHEM + LettuceDetect, FULL, both sets
!python m3_implementation/test_result/hallucination_result/external_baselines/run_external_baselines.py \
  --test-set m3_implementation/test_result/hallucination_result/expanded_eval/labeled_test_set_expanded.jsonl \
  --tools hhem,lettuce \
  --out m3_implementation/test_result/hallucination_result/expanded_eval/colab_results_external_standard.json
!python m3_implementation/test_result/hallucination_result/external_baselines/run_external_baselines.py \
  --test-set m3_implementation/test_result/hallucination_result/expanded_eval/hard_set/labeled_hard_set.jsonl \
  --tools hhem,lettuce \
  --out m3_implementation/test_result/hallucination_result/expanded_eval/colab_results_external_hard.json

In [ ]:
# 7 — SummaC, FULL, both sets (GPU makes full runs feasible)
!python m3_implementation/test_result/hallucination_result/external_baselines/run_external_baselines.py \
  --test-set m3_implementation/test_result/hallucination_result/expanded_eval/labeled_test_set_expanded.jsonl \
  --tools summac \
  --out m3_implementation/test_result/hallucination_result/expanded_eval/colab_results_summac_standard.json
!python m3_implementation/test_result/hallucination_result/external_baselines/run_external_baselines.py \
  --test-set m3_implementation/test_result/hallucination_result/expanded_eval/hard_set/labeled_hard_set.jsonl \
  --tools summac \
  --out m3_implementation/test_result/hallucination_result/expanded_eval/colab_results_summac_hard.json

In [ ]:
# 8 — Package all colab results and download
import glob, zipfile
EE = 'm3_implementation/test_result/hallucination_result/expanded_eval'
outs = glob.glob(f'{EE}/colab_results_*.json')
print('\n'.join(outs))
with zipfile.ZipFile('colab_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in outs:
        z.write(p, os.path.basename(p))
from google.colab import files as gfiles
gfiles.download('colab_results.zip')